# Backbone Experiment: convnext_base

## Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # repo root
from config import Config
from data.cifake import get_cifake_loaders
from experiments.train import train
from experiments.evaluate import full_evaluation
from experiments.baseline_spatial_only import run_spatial_only_baseline
from tqdm.notebook import tqdm
import pandas as pd
import torch

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


Device: cuda
GPU:    NVIDIA RTX PRO 4000 Blackwell
VRAM:   25.2 GB


## Shared Config

In [2]:
# Only line to change
BACKBONE = "convnext_base"

def make_cfg(fusion_mode, frozen=False):
    cfg = Config()
    cfg.data.cifake_root    = "../data/raw/cifake"
    cfg.data.num_workers    = 4 # change to 0 if on Windows
    cfg.train.backbone_lr   = 1e-4  
    cfg.train.lr            = 1e-4
    cfg.data.batch_size     = 64
    cfg.data.jpeg_aug       = True
    cfg.backbone.name       = BACKBONE
    cfg.backbone.pretrained = True
    cfg.backbone.frozen     = frozen
    cfg.backbone.img_size   = cfg.data.image_size  
    cfg.fusion.mode         = fusion_mode
    cfg.train.epochs        = 50
    cfg.experiment_name     = f"{BACKBONE}_{fusion_mode}{'_frozen' if frozen else ''}"
    cfg.notes               = f"CIFAKE · {BACKBONE} · {fusion_mode}{'_frozen' if frozen else ''}"
    return cfg

# Load data
cfg_data = make_cfg("joint_only") # backbone settings only, fusion mode ignored
train_loader, val_loader, test_loader = get_cifake_loaders(cfg_data)
print(f"Train: {len(train_loader.dataset):,}  Val: {len(val_loader.dataset):,}  Test: {len(test_loader.dataset):,}")

Train: 85,000  Val: 15,000  Test: 20,000
Train: 85,000  Val: 15,000  Test: 20,000


## Experiment 0: Spatial-Only Baseline
Trains only the spatial backbone as a standalone classifier with no frequency branch.
This gives the honest floor. How well the backbone alone can classify real vs fake.
All delta values in later experiments are relative to this number.


In [3]:
cfg0 = make_cfg("joint_only")  # backbone settings only, fusion mode ignored
cfg0.experiment_name = f"{BACKBONE}_spatial_only"
cfg0.notes           = f"spatial-only baseline, no freq branch, {BACKBONE}"
spatial_acc = run_spatial_only_baseline(cfg0, train_loader, val_loader, test_loader)
print(f"\nSpatial-only floor: {spatial_acc:.1%}")
print("All subsequent delta values are relative to this number.")

Device: cuda
Training spatial-only baseline (convnext_base) for 50 epochs...
Train: 85,000  Val: 15,000


Epoch   1/50 | train_loss=0.1226 | val_acc=97.3%


Epoch   2/50 | train_loss=0.0690 | val_acc=97.3%


Epoch   3/50 | train_loss=0.0520 | val_acc=98.2%


Epoch   4/50 | train_loss=0.0418 | val_acc=98.3%


...

Epoch  46/50 | train_loss=0.0002 | val_acc=98.7%


Epoch  47/50 | train_loss=0.0001 | val_acc=98.7%


Epoch  48/50 | train_loss=0.0001 | val_acc=98.7%


Epoch  49/50 | train_loss=0.0002 | val_acc=98.6%


Epoch  50/50 | train_loss=0.0002 | val_acc=98.7%

Spatial-only results (convnext_base):
  Accuracy: 98.7%
  AUC-ROC:  0.998
  F1:       0.987
Results saved → ./results/results.csv  (convnext_base_spatial_only, acc=0.9872)
Results saved to ./results/results.csv

Spatial-only floor: 98.7%
All subsequent delta values are relative to this number.


## Experiment 1: Joint-Only Baseline
Both branches concatenated into a single shared classifier. No weighting between branches.
This is the floor — all other experiments are compared against this delta value.


In [4]:
cfg1 = make_cfg("joint_only")
print(f"Running: {cfg1.experiment_name}")
train(cfg1, train_loader, val_loader, test_loader)

Running: convnext_base_joint_only
Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell

Experiment: convnext_base_joint_only
Backbone: convnext_base | Fusion: joint_only | Frozen: False
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.3626 | val_acc=97.5% | val_auc=0.997 | val_f1=0.975
  grad norms — freq=0.9260 | spatial=0.3733
  -> Saved best val_acc=97.5%


Epoch   2/50 | train_loss=0.2566 | val_acc=97.7% | val_auc=0.998 | val_f1=0.977
  -> Saved best val_acc=97.7%


Epoch   3/50 | train_loss=0.2166 | val_acc=97.9% | val_auc=0.998 | val_f1=0.979
  -> Saved best val_acc=97.9%


Epoch   4/50 | train_loss=0.1954 | val_acc=98.4% | val_auc=0.999 | val_f1=0.984
  -> Saved best val_acc=98.4%


...

Epoch  47/50 | train_loss=0.0784 | val_acc=98.8% | val_auc=0.998 | val_f1=0.988


Epoch  48/50 | train_loss=0.0782 | val_acc=98.8% | val_auc=0.998 | val_f1=0.988


Epoch  49/50 | train_loss=0.0778 | val_acc=98.8% | val_auc=0.998 | val_f1=0.988


Epoch  50/50 | train_loss=0.0778 | val_acc=98.8% | val_auc=0.998 | val_f1=0.988

Training complete. Best val accuracy: 98.8%
Results will be logged to CSV after full_evaluation().


0.988

In [5]:
results1 = full_evaluation(
    cfg1,
    checkpoint_path=f"./checkpoints/best_{cfg1.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_convnext_base_joint_only.pt

EVALUATION — convnext_base_joint_only
Backbone: convnext_base | Fusion: joint_only | Frozen: False
  Joint accuracy:     98.8%
  AUC-ROC:            0.999
  F1:                 0.988
  Spatial-only:       98.8%
  Freq-only:          93.1%
  Delta (Δ):          +0.0%  (freq branch contribution)
Results saved → ./results/results.csv  (convnext_base_joint_only, acc=0.9876)


## Experiment 2: Scalar Fusion
Two learned scalars (a, b) weight each branch. Softmax ensures a+b=1 always.
Watch the scalar values logged each epoch. `b`is the frequency weight.
If `b` drops below 0.1 early in training the frequency branch is being ignored.


In [6]:
cfg2 = make_cfg("scalar")
print(f"Running: {cfg2.experiment_name}")
train(cfg2, train_loader, val_loader, test_loader)

Running: convnext_base_scalar
Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell

Experiment: convnext_base_scalar
Backbone: convnext_base | Fusion: scalar | Frozen: False
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.3637 | val_acc=96.9% | val_auc=0.998 | val_f1=0.970
  scalars — spatial=0.498 freq=0.502
  grad norms — freq=0.3691 | spatial=0.9268
  -> Saved best val_acc=96.9%


Epoch   2/50 | train_loss=0.2563 | val_acc=98.1% | val_auc=0.999 | val_f1=0.981
  scalars — spatial=0.496 freq=0.504
  -> Saved best val_acc=98.1%


Epoch   3/50 | train_loss=0.2193 | val_acc=97.4% | val_auc=0.998 | val_f1=0.974
  scalars — spatial=0.495 freq=0.505


Epoch   4/50 | train_loss=0.1999 | val_acc=98.3% | val_auc=0.999 | val_f1=0.983
  scalars — spatial=0.494 freq=0.506
  -> Saved best val_acc=98.3%


...

Epoch  47/50 | train_loss=0.0877 | val_acc=98.7% | val_auc=0.998 | val_f1=0.987
  scalars — spatial=0.483 freq=0.517


Epoch  48/50 | train_loss=0.0875 | val_acc=98.7% | val_auc=0.998 | val_f1=0.987
  scalars — spatial=0.483 freq=0.517


Epoch  49/50 | train_loss=0.0877 | val_acc=98.7% | val_auc=0.998 | val_f1=0.987
  scalars — spatial=0.483 freq=0.517


Epoch  50/50 | train_loss=0.0868 | val_acc=98.8% | val_auc=0.998 | val_f1=0.988
  scalars — spatial=0.483 freq=0.517

Training complete. Best val accuracy: 98.8%
Results will be logged to CSV after full_evaluation().


0.9877333333333334

In [7]:
results2 = full_evaluation(
    cfg2,
    checkpoint_path=f"./checkpoints/best_{cfg2.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_convnext_base_scalar.pt

EVALUATION — convnext_base_scalar
Backbone: convnext_base | Fusion: scalar | Frozen: False
  Joint accuracy:     98.7%
  AUC-ROC:            0.999
  F1:                 0.987
  Spatial-only:       98.7%
  Freq-only:          91.6%
  Delta (Δ):          +0.0%  (freq branch contribution)

  No warning signs triggered. ✓
Results saved → ./results/results.csv  (convnext_base_scalar, acc=0.9868)


## Experiment 3: Gating Fusion 
A small MLP outputs a per-image gate value in [0,1] controlling how much to trust the frequency branch.
Key metric beyond accuracy: **gate entropy must be > 0.3 nats**.
Below that the gate is outputting near-constant values and is not genuinely adapting per sample.


In [8]:
cfg3 = make_cfg("gating")
print(f"Running: {cfg3.experiment_name}")
train(cfg3, train_loader, val_loader, test_loader)

Running: convnext_base_gating
Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell



Experiment: convnext_base_gating
Backbone: convnext_base | Fusion: gating | Frozen: False
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.4050 | val_acc=97.8% | val_auc=0.998 | val_f1=0.978
  gate — entropy=1.431 nats | mean=0.436 | var=0.0025
  grad norms — freq=0.9540 | spatial=0.2970
  -> Saved best val_acc=97.8%


Epoch   2/50 | train_loss=0.3243 | val_acc=97.2% | val_auc=0.998 | val_f1=0.973
  gate — entropy=0.935 nats | mean=0.472 | var=0.0008


Epoch   3/50 | train_loss=0.2805 | val_acc=97.7% | val_auc=0.998 | val_f1=0.977
  gate — entropy=1.272 nats | mean=0.474 | var=0.0025


...

Epoch  47/50 | train_loss=0.1757 | val_acc=98.8% | val_auc=0.998 | val_f1=0.988
  gate — entropy=1.086 nats | mean=0.263 | var=0.0020


Epoch  48/50 | train_loss=0.1833 | val_acc=98.8% | val_auc=0.998 | val_f1=0.988
  gate — entropy=0.986 nats | mean=0.257 | var=0.0018


Epoch  49/50 | train_loss=0.1873 | val_acc=98.8% | val_auc=0.998 | val_f1=0.988
  gate — entropy=0.935 nats | mean=0.255 | var=0.0017


Epoch  50/50 | train_loss=0.1935 | val_acc=98.8% | val_auc=0.998 | val_f1=0.988
  gate — entropy=0.926 nats | mean=0.254 | var=0.0017

Training complete. Best val accuracy: 98.8%
Results will be logged to CSV after full_evaluation().


0.9884666666666667

In [9]:
results3 = full_evaluation(
    cfg3,
    checkpoint_path=f"./checkpoints/best_{cfg3.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_convnext_base_gating.pt

EVALUATION — convnext_base_gating
Backbone: convnext_base | Fusion: gating | Frozen: False
  Joint accuracy:     98.9%
  AUC-ROC:            0.999
  F1:                 0.989
  Spatial-only:       98.8%
  Freq-only:          92.4%
  Delta (Δ):          +0.0%  (freq branch contribution)

  Gate distribution:
    entropy:  0.959 nats  (OK)
    mean:     0.304
    variance: 0.0014

  No warning signs triggered. ✓
Results saved → ./results/results.csv  (convnext_base_gating, acc=0.9888)


## Experiment 4: Gating Fusion, Frozen Backbone
Same as Experiment 3 but backbone weights are locked.
Only the projection head, frequency branch, fusion, and joint head train.

If the frequency branch helps more here than in Experiment 3, it means the backbone
was learning to capture spectral information during fine-tuning in Experiment 3, essentially teaching itself what the frequency branch provides.


In [10]:
cfg4 = make_cfg("gating", frozen=True)
print(f"Running: {cfg4.experiment_name}")
train(cfg4, train_loader, val_loader, test_loader)

Running: convnext_base_gating_frozen
Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell

Experiment: convnext_base_gating_frozen
Backbone: convnext_base | Fusion: gating | Frozen: True
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.5124 | val_acc=92.6% | val_auc=0.983 | val_f1=0.923
  gate — entropy=2.193 nats | mean=0.491 | var=0.0118
  grad norms — freq=0.9672 | spatial=0.2370
  -> Saved best val_acc=92.6%


Epoch   2/50 | train_loss=0.4270 | val_acc=92.6% | val_auc=0.985 | val_f1=0.930
  gate — entropy=2.223 nats | mean=0.508 | var=0.0126
  -> Saved best val_acc=92.6%


Epoch   3/50 | train_loss=0.3942 | val_acc=94.3% | val_auc=0.987 | val_f1=0.945
  gate — entropy=2.259 nats | mean=0.538 | var=0.0136
  -> Saved best val_acc=94.3%


Epoch   4/50 | train_loss=0.3759 | val_acc=95.2% | val_auc=0.990 | val_f1=0.953
  gate — entropy=2.429 nats | mean=0.552 | var=0.0195
  -> Saved best val_acc=95.2%


...

Epoch  47/50 | train_loss=0.2206 | val_acc=96.4% | val_auc=0.995 | val_f1=0.965
  gate — entropy=2.783 nats | mean=0.569 | var=0.0473


Epoch  48/50 | train_loss=0.2181 | val_acc=96.4% | val_auc=0.995 | val_f1=0.964
  gate — entropy=2.789 nats | mean=0.566 | var=0.0479


Epoch  49/50 | train_loss=0.2214 | val_acc=96.5% | val_auc=0.995 | val_f1=0.966
  gate — entropy=2.787 nats | mean=0.569 | var=0.0478


Epoch  50/50 | train_loss=0.2195 | val_acc=96.5% | val_auc=0.995 | val_f1=0.966
  gate — entropy=2.784 nats | mean=0.573 | var=0.0477

Training complete. Best val accuracy: 96.6%
Results will be logged to CSV after full_evaluation().


0.9664

In [11]:
results4 = full_evaluation(
    cfg4,
    checkpoint_path=f"./checkpoints/best_{cfg4.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_convnext_base_gating_frozen.pt

EVALUATION — convnext_base_gating_frozen
Backbone: convnext_base | Fusion: gating | Frozen: True
  Joint accuracy:     96.5%
  AUC-ROC:            0.995
  F1:                 0.965
  Spatial-only:       90.8%
  Freq-only:          90.8%
  Delta (Δ):          +5.7%  (freq branch contribution)

  Gate distribution:
    entropy:  2.796 nats  (OK)
    mean:     0.584
    variance: 0.0502

  No warning signs triggered. ✓
Results saved → ./results/results.csv  (convnext_base_gating_frozen, acc=0.9653)


## Summary: convnext_base
Compare all four runs. Key columns: accuracy, delta (freq branch contribution), gate_entropy.


In [12]:
df = pd.read_csv("./results/results.csv")
mask = df["backbone"] == BACKBONE
cols = ["experiment_name", "fusion", "frozen", "accuracy", "auc_roc", "f1", "gate_entropy"]
print(df[mask][cols].to_string(index=False))

            experiment_name     fusion  frozen  accuracy  auc_roc     f1  gate_entropy
         freq_only_baseline     gating   False    0.9436   0.9871 0.9446        0.0000
 convnext_base_spatial_only joint_only   False    0.9872   0.9983 0.9872        0.0000
   convnext_base_joint_only joint_only   False    0.9876   0.9985 0.9876        0.0000
       convnext_base_scalar     scalar   False    0.9868   0.9985 0.9868        0.0000
       convnext_base_gating     gating   False    0.9888   0.9987 0.9888        0.9590
convnext_base_gating_frozen     gating    True    0.9653   0.9948 0.9654        2.7965


**What to look for:**
- **Delta** (printed by full_evaluation) - how much the frequency branch added over spatial alone
- **Gate entropy** - must be > 0.3 nats for gating experiments to be valid
- **Frozen vs fine-tuned** - if frozen delta > fine-tuned delta, the backbone was absorbing spectral information during fine-tuning
